# 05 · Smoke test of the one-shot local runner
Team Dropouts · Correction Inception

## Setup
Kaggle settings: **Accelerator = GPU T4 x2**, **Internet = On**. Add-ons → Secrets → `HF_TOKEN` (you must have accepted FERMAT's terms on Hugging Face).

Add Inputs: datasets **lma-src**, **edu-text**, **edu-circuit-hw-images**, and (to reuse cached grades) the **output of notebook 03**.

In [ ]:
import os, glob, shutil, zipfile, sys
IN = os.environ.get('KAGGLE_INPUT', '/kaggle/input')
WORK = os.environ.get('KAGGLE_WORK', '/kaggle/working')
def find(pattern):
    hits = sorted(glob.glob(f'{IN}/**/{pattern}', recursive=True) + glob.glob(f'{WORK}/_in/**/{pattern}', recursive=True))
    return hits[0] if hits else None
# Kaggle usually unzips uploads; if it kept a .zip, extract it ourselves
if not (find('src/eval/llm.py') and find('edu_data') and find('Screenshot_output_anon/Homework*')):
    for z in glob.glob(f'{IN}/**/*.zip', recursive=True):
        zipfile.ZipFile(z).extractall(f'{WORK}/_in/' + os.path.basename(z)[:-4])
llm, EDU_DATA, EDU_REPO = find('src/eval/llm.py'), find('edu_data'), find('edu_repo')
hw = find('Screenshot_output_anon/Homework*')
IMAGES = os.path.dirname(os.path.dirname(hw)) if hw else None
if not (llm and EDU_DATA and EDU_REPO and IMAGES):
    for root, dirs, files in os.walk(IN):
        if root.count(os.sep) - IN.count(os.sep) < 4: print(root, dirs[:5], files[:5])
    raise SystemExit('Missing inputs: attach lma-src, edu-text AND the images dataset via Add Input. Found: '
                     + str(dict(src=llm, edu_data=EDU_DATA, edu_repo=EDU_REPO, images=IMAGES)))
shutil.copytree(os.path.dirname(os.path.dirname(llm)), f'{WORK}/src', dirs_exist_ok=True)
os.environ.update(EDU_DATA=EDU_DATA, EDU_REPO=EDU_REPO)
os.chdir(WORK)
for rel in ['results/llm_cache.sqlite']:  # reuse grades cached by notebook 03 (attach its output as an input)
    old = find(rel)
    if old and not os.path.exists(f'{WORK}/{rel}'):
        os.makedirs(os.path.dirname(f'{WORK}/{rel}'), exist_ok=True); shutil.copy(old, f'{WORK}/{rel}'); print('reusing cache', old)
DRY = bool(os.environ.get('DRY_JUDGE'))  # local dry run: mock models, no GPU
if not DRY:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    os.system(f'{sys.executable} -m pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils datasets python-dotenv')
    os.system('nvidia-smi --query-gpu=name,memory.total --format=csv')
READER = 'mock' if DRY else 'hf:Qwen/Qwen2.5-VL-3B-Instruct'
GRADER = 'mock' if DRY else 'hf-text:Qwen/Qwen2.5-7B-Instruct'
os.environ.update(HF_BATCH='8', HF_TOKEN_BUDGET='16000')
LIMIT = os.environ.get('DRY_LIMIT', '')
ITEMS = 'src/eval/items_with_expert_transcript.csv'  # the 293 exams that have an expert transcript
print('code:', llm, '| data:', EDU_DATA, '| images:', IMAGES)

## Smoke test of `scripts/run_all.py` on a real GPU (3 samples per step, ~30-40 min incl. downloads)
Same script that runs the full job on the local RTX 5070 Ti. Must end with **ALL DONE**.

In [ ]:
CODE = os.path.dirname(os.path.dirname(os.path.dirname(llm)))  # dataset root holding src/ and scripts/
shutil.copytree(f'{CODE}/scripts', f'{WORK}/scripts', dirs_exist_ok=True)
shutil.copy(f'{CODE}/requirements-gpu.txt', WORK)
if not DRY:
    os.system(f'{sys.executable} -m pip install -q -r {WORK}/requirements-gpu.txt')
rc = os.system(f'{sys.executable} {WORK}/scripts/run_all.py --edu-data {EDU_DATA} --edu-repo {EDU_REPO} '
               f'--images {IMAGES} --limit 3')
print('exit code', rc)
if rc: raise SystemExit('run_all reported problems: read the PREFLIGHT / FAILED lines above')

In [ ]:
# results + transcripts (markdown only, not the copied page images)
with zipfile.ZipFile(f'{WORK}/smoke_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f'{WORK}/results/**/*', recursive=True) + glob.glob(f'{WORK}/Outputs/**/*.md', recursive=True):
        if os.path.isfile(f): z.write(f, os.path.relpath(f, WORK))
print(os.path.getsize(f'{WORK}/smoke_results.zip') // 1024, 'KB')